In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Deterministic Keyword Based

In [2]:
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
BLOCKED: How do I hack into a database?
ALLOWED: What is the capital of France?
BLOCKED: Explain how malware spreads


### LLM Model Based

In [6]:
from langchain_openai import ChatOpenAI

def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads


### PII Dection Middleware (Redacting)

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model="gpt-4o",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

ImportError: cannot import name 'create_agent' from 'langchain.agents' (C:\Users\hitli\AppData\Roaming\Python\Python312\site-packages\langchain\agents\__init__.py)

In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is john.doe@example.com and my card is "
            "5105-1051-0510-5100. Can you help me?"
        )
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I'm unable to assist you with card or email issues directly due to privacy and security reasons. However, if you have a specific question or request related to your account or a service, please let me know how I can assist you further!


In [ ]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


### Human In The Loop

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

hitl_agent = create_agent(
    model="gpt-4o",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,    # Require approval
                "search_web": False,       # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

In [ ]:
# Step 1: Invoke -- agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused -- awaiting human approval ===")

=== Agent paused -- awaiting human approval ===


In [ ]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
Could you please provide the subject and body of the email you'd like me to send?


In [ ]:
# Alternative -- Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)


=== Rejected! Final response ===
Looks like you decided against deleting the records. If there's anything else you need, feel free to ask!


### Custom Before-Agent Guardrail (Input evaluation)

In [ ]:
from typing import Any
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, hook_config
)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything --
    zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked -- keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing "
                            "inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=[
                "hack", "exploit", "malware", "jailbreak", "bypass"
            ]
        ),
    ],
)

In [ ]:
# Test 1: Safe request -- should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("Safe request response:")
print(result["messages"][-1].content)

Safe request response:
Machine learning is a subset of artificial intelligence (AI) that involves the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. Instead of being programmed with specific instructions for every scenario, a machine learning system learns from and makes predictions or decisions based on data. The learning process involves feeding vast amounts of data to the system, which it then uses to detect patterns or make decisions.

There are several types of machine learning, including:

1. **Supervised Learning**: Involves training a model on labeled data, meaning the input comes with the correct output. The goal is for the model to learn to map inputs to the correct outputs.

2. **Unsupervised Learning**: Involves training on data that has no labels; the system tries to learn the patterns and structure from the data itself. It's often used for clustering or association tasks.

3. **Semi-supervised Learnin

In [ ]:
# Test 2: Unsafe request -- should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("Unsafe request response:")
print(result["messages"][-1].content)

Blocked -- keyword detected: 'hack'
Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


### Custom After Agent Guardrail 

In [ ]:
from typing import Any
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, hook_config
)
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe \
and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke(
            [{"role": "user", "content": safety_prompt}]
        )

        if "UNSAFE" in result.content.upper():
            print("Output flagged as UNSAFE -- replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="gpt-4o",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

In [ ]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
To provide the weather information, I need to know your specific location. Please share your city or region, and I'll find the weather details for today.


### Layered/combined Guardrails

User Input
    |
[Layer 1] ContentFilterMiddleware     -- Deterministic input filter
    |
[Layer 2] PIIMiddleware (input)       -- PII redaction on input
    |
[Layer 3] HumanInTheLoopMiddleware    -- Approval for sensitive tools
    |
[Layer 4] PIIMiddleware (output)      -- PII redaction on output
    |
[Layer 5] SafetyGuardrailMiddleware   -- Model-based output safety
    |
User Response

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    PIIMiddleware, HumanInTheLoopMiddleware
)
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware"]
        ),

        # Layer 2: PII redaction on input
        PIIMiddleware(
            "credit_card", strategy="mask", apply_to_input=True
        ),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": True,
                "search_tool": False,
            }
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware(
            "email", strategy="redact", apply_to_output=True
        ),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("Production-grade agent with 5-layer guardrails created!")

Production-grade agent with 5-layer guardrails created!


## Healthcare Chatbot


### Healthcare specific content filter

In [ ]:
from typing import Any
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, hook_config
)
from langchain.agents.middleware import (
    PIIMiddleware, HumanInTheLoopMiddleware
)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage

class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = [
        "drug synthesis", "self-harm", "suicide method",
        "weapon", "hack"
    ]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help "
                            "with medical questions, appointments, and "
                            "health information. If you're in crisis, "
                            "please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None

### Medical Output Validator

In [ ]:
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = (
        "\n\nThis is general health information, not medical advice. "
        "Please consult a qualified healthcare professional."
    )

    @hook_config(can_jump_to=["end"])
    def after_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None

### Healthcare Tools

In [ ]:
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return (
        f"Symptom information for: {symptoms}. "
        "Please consult a doctor for diagnosis."
    )

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return (
        f"Appointment booked for {patient_name} "
        f"with Dr. {doctor} on {date}"
    )

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return (
        f"General info about {medication}. "
        "Always follow your doctor's prescription."
    )

### Assembling the Helthcare Chatbot

In [ ]:
healthcare_bot = create_agent(
    model="gpt-4o",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware(
            "credit_card", strategy="mask", apply_to_input=True
        ),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, "
        "and help book appointments. Always be empathetic and "
        "remind users to consult a doctor for diagnosis."
    ),
)

print("Healthcare chatbot with full guardrail stack created!")

Healthcare chatbot with full guardrail stack created!


### Testing the Healthcare Chatbot

In [ ]:
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)
print(result["messages"][-1].content)

Type 2 Diabetes can present with various symptoms. I will look up the specific symptoms for you. Please give me a moment.
Type 2 Diabetes symptoms can include:

1. Increased thirst and frequent urination
2. Excessive hunger
3. Fatigue
4. Blurred vision
5. Slow-healing sores
6. Frequent infections (e.g., gums, skin, vaginal, or bladder)
7. Unintended weight loss

These symptoms can vary from person to person, and it's crucial to consult a doctor for a proper diagnosis if you suspect Type 2 Diabetes or experience any of these symptoms.

This is general health information, not medical advice. Please consult a qualified healthcare professional.


In [ ]:
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is patient123@gmail.com. "
            "What can I take for a headache?"
        )
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
For a headache, there are several over-the-counter options you might consider, but it's important to always follow any prescription or advice given by your doctor. Common medications include:

1. **Acetaminophen (Tylenol)**: Often used for mild to moderate headaches.
2. **Ibuprofen (Advil, Motrin)**: A nonsteroidal anti-inflammatory drug (NSAID) that can be more effective for certain types of headaches.
3. **Aspirin**: Another NSAID that can help with pain relief.
4. **Naproxen (Aleve)**: Also a NSAID which can be effective for headache relief.

Remember, if your headaches are severe, frequent, or worsening, it's important to consult with a healthcare professional to determine the underlying cause and appropriate treatment. Also, follow the instructions on the medication label or those from your healthcare provider when taking any medication.

This is general health information, not medical advice. Please consult a qualified healthcare professional.


In [ ]:
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
    config=config_t1
)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

=== Blocked Request ===
I'm really sorry, but I can't assist with that request. Synthesizing drugs at home is highly dangerous and illegal in many places. It requires specialized knowledge, equipment, and safety procedures that are not feasible outside of professional, controlled environments. It's always important to obtain medications through proper, legal channels and to use them in accordance with medical advice from a healthcare professional. If you're having trouble accessing the medication you need, consider speaking with a doctor or a pharmacist for guidance.


In [ ]:
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking -- Awaiting Approval ===")

# Approve
from langgraph.types import Command

approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("=== After Approval ===")
print(approved["messages"][-1].content)

=== Appointment Booking -- Awaiting Approval ===
=== After Approval ===
Could you please provide your full name to proceed with booking the appointment with Dr. Sharma on March 15?


lets say for example, in manufacturing sector BOM(Bill of material), I have set Keyword BOM as need to be blocked. like in some cases like how to create a bom (allowed). and how to destroy bom(blocked). how to do this and how it is evaluated


What you’re describing is a **context-sensitive policy**, not a simple keyword block. A naive “block `BOM`” rule will fail because *BOM itself is not harmful*—intent is encoded in surrounding language.

You need **intent-aware filtering**, typically implemented as a two-stage middleware.

---

# 1) Problem Decomposition

You have:

* Term: **“BOM” (Bill of Materials)**
* Allowed intent: *create / manage / view*
* Blocked intent: *destroy / sabotage / delete improperly*

So the policy is:

```id="policy"
IF keyword == "BOM"
AND intent == destructive
→ BLOCK
ELSE → ALLOW
```

---

# 2) How to Implement in LangChain

## Approach A — Rule-based (No LLM)

You classify intent using verb patterns.

### Example logic

```python
ALLOWED_VERBS = ["create", "generate", "build", "design", "view"]
BLOCKED_VERBS = ["destroy", "delete", "corrupt", "tamper", "remove"]

def classify_bom_intent(text):
    text_lower = text.lower()

    if "bom" in text_lower:
        if any(v in text_lower for v in BLOCKED_VERBS):
            return "blocked"
        elif any(v in text_lower for v in ALLOWED_VERBS):
            return "allowed"

    return "neutral"
```

### Evaluation

| Input                      | Result          |
| -------------------------- | --------------- |
| "How to create a BOM?"     | Allowed         |
| "How to destroy BOM data?" | Blocked         |
| "Explain BOM structure"    | Neutral/Allowed |

### Limitation

* Fails on paraphrases:

  * “wipe BOM records”
  * “invalidate BOM entries”

---

## Approach B — LLM-based Intent Classification (Recommended)

You insert a middleware step that **calls an LLM classifier before the main chain**.

### Step 1: Intent classification prompt

```text
Classify the intent of the following query related to BOM:

Return one label:
- SAFE: normal usage (create, view, manage)
- DESTRUCTIVE: attempts to delete, corrupt, sabotage
- UNKNOWN

Query: {input}
```

### Step 2: Middleware flow

```id="flow"
User Input
   ↓
Middleware → LLM (intent classifier)
   ↓
IF DESTRUCTIVE → Block
ELSE → Pass to main LLM
```

### Example outputs

| Query                  | Classification |
| ---------------------- | -------------- |
| "How to create a BOM?" | SAFE           |
| "How to destroy BOM?"  | DESTRUCTIVE    |
| "How to modify BOM?"   | SAFE           |

---

## 3) Hybrid (Best Practice)

Combine both:

### Layered filtering

1. **Fast rule-based pre-check**
2. **LLM fallback for ambiguity**

```python
def middleware(input_text):
    result = classify_bom_intent(input_text)

    if result == "blocked":
        return "Blocked by policy"

    if result == "neutral":
        # call LLM classifier
        intent = llm_classify(input_text)
        if intent == "DESTRUCTIVE":
            return "Blocked by policy"

    return input_text
```

---

## 4) Where This Runs in LangChain

In **LangChain**, you’d typically implement this as:

* `RunnableLambda` (pre-processing middleware)
* Or a custom chain wrapper

### LCEL-style pipeline

```python
chain = (
    RunnableLambda(middleware)
    | main_llm_chain
)
```

---

## 5) Important Design Considerations

### A) Context Expansion

Users won’t always say “BOM” explicitly:

* “material list”
* “assembly components”

→ Maintain a **synonym map**

---

### B) False Positives

Example:

* “delete duplicate BOM entries” (valid ERP operation)

→ You need:

* **fine-grained intent labels**

  * destructive vs administrative deletion

---

### C) Domain Awareness

In manufacturing systems (SAP, PLM tools):

* “delete BOM” may be legitimate with authorization

→ Policy should include:

* user role
* system context

---

## 6) Key Insight

* Keyword filtering = **lexical**
* What you need = **semantic intent classification**

So:

> Middleware **may call an LLM**, but only when rules are insufficient.

---

## If You Want to Go Further

I can help you design:

* A production-grade policy engine (with roles + audit logs)
* A prompt-injection-resistant classifier
* Or a full LangChain LCEL implementation with tracing

Just specify the level of rigor you need.
